<a href="https://colab.research.google.com/github/hridayrjain/BIS-Lab/blob/main/BIS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hriday R Jain

1BM24CS119

Algorithm : Gene Algorithm for optimization problem




Application : Cyber Attack detection Optimization



Select features and tune model parameters for network intrusion detection



Objective : Improve detection performance by reducing false alarms and computation cost


In [4]:
import numpy as np
import pandas as pd
import random
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

# 1. Define Realistic Cyber Network Features
feature_names = [
    "duration", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent",
    "hot", "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate"
]

# Generate simulated network traffic dataset (normal vs. attack)
X, y = make_classification(
    n_samples=2000,
    n_features=len(feature_names),
    n_informative=15,
    n_classes=2,
    random_state=42
)

# Split into training and test validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

NUM_FEATURES = len(feature_names)
PARAM_BITS = 6
CHROMOSOME_LENGTH = NUM_FEATURES + PARAM_BITS

POP_SIZE = 20
GENERATIONS = 10
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

def decode_chromosome(chromosome):
    feature_mask = chromosome[:NUM_FEATURES].astype(bool)
    param_bits = chromosome[NUM_FEATURES:]

    # Decode 3 bits for n_estimators (Range: 10 to 150)
    n_est_bits = "".join(str(int(b)) for b in param_bits[:3])
    n_estimators = 10 + int(n_est_bits, 2) * 20

    # Decode 3 bits for max_depth (Range: 3 to 24)
    depth_bits = "".join(str(int(b)) for b in param_bits[3:])
    max_depth = 3 + int(depth_bits, 2) * 3

    return feature_mask, n_estimators, max_depth

def calculate_fitness(chromosome):
    feature_mask, n_estimators, max_depth = decode_chromosome(chromosome)

    # If no features are selected, penalize with zero fitness
    if not any(feature_mask):
        return 0.0

    X_tr_selected = X_train[:, feature_mask]
    X_ts_selected = X_test[:, feature_mask]

    # Train detector on selected network features
    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    clf.fit(X_tr_selected, y_train)
    predictions = clf.predict(X_ts_selected)

    # Evaluate Intrusion Detection metrics
    f1 = f1_score(y_test, predictions, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

    # False Alarm Rate (critical for network operations to prevent disruption)
    far = fp / (fp + tn) if (fp + tn) > 0 else 1.0

    # Feature overhead ratio (smaller feature subsets mean faster real-time execution)
    feature_ratio = np.sum(feature_mask) / NUM_FEATURES

    # Objective function weighting
    fitness = (0.6 * f1) + (0.2 * (1 - far)) + (0.2 * (1 - feature_ratio))
    return fitness

def initialize_population():
    return np.random.randint(2, size=(POP_SIZE, CHROMOSOME_LENGTH))

def selection(population, fitnesses):
    total_fit = np.sum(fitnesses)
    if total_fit == 0:
        probs = np.ones(POP_SIZE) / POP_SIZE
    else:
        probs = fitnesses / total_fit
    selected_indices = np.random.choice(POP_SIZE, size=POP_SIZE, p=probs)
    return population[selected_indices]

def crossover(parent1, parent2):
    if random.random() < CROSSOVER_RATE:
        point = random.randint(1, CHROMOSOME_LENGTH - 1)
        child1 = np.concatenate((parent1[:point], parent2[point:]))
        child2 = np.concatenate((parent2[:point], parent1[point:]))
        return child1, child2
    return parent1.copy(), parent2.copy()

def mutate(chromosome):
    for i in range(CHROMOSOME_LENGTH):
        if random.random() < MUTATION_RATE:
            chromosome[i] = 1 - chromosome[i]
    return chromosome

# Initialize and execute Genetic Algorithm
population = initialize_population()
best_chromosome = None
best_fitness = -1

print(f"{'Gen':<6}{'Best Fitness':<15}{'Features Selected':<20}{'n_estimators':<15}{'max_depth'}")
print("-" * 65)

for generation in range(GENERATIONS):
    fitnesses = np.array([calculate_fitness(ind) for ind in population])

    max_idx = np.argmax(fitnesses)
    if fitnesses[max_idx] > best_fitness:
        best_fitness = fitnesses[max_idx]
        best_chromosome = population[max_idx].copy()

    feat_mask, n_est, depth = decode_chromosome(population[max_idx])
    print(f"{generation+1:<6}{fitnesses[max_idx]:<15.4f}{np.sum(feat_mask):<20}{n_est:<15}{depth}")

    next_population = selection(population, fitnesses)

    for i in range(0, POP_SIZE, 2):
        if i+1 < POP_SIZE:
            child1, child2 = crossover(next_population[i], next_population[i+1])
            next_population[i] = mutate(child1)
            next_population[i+1] = mutate(child2)

    population = next_population

best_features, best_n_est, best_depth = decode_chromosome(best_chromosome)
selected_feature_names = [feature_names[i] for i, val in enumerate(best_features) if val]

print("\n" + "="*50 + "\nCYBER ATTACK DETECTION OPTIMIZATION COMPLETE\n" + "="*50)
print(f"Optimal Fitness Score: {best_fitness:.4f}")
print(f"Optimized Hyperparameters: n_estimators={best_n_est}, max_depth={best_depth}")
print(f"Selected Cyber Features ({len(selected_feature_names)}/{NUM_FEATURES}): {selected_feature_names}")

Gen   Best Fitness   Features Selected   n_estimators   max_depth
-----------------------------------------------------------------
1     0.7700         18                  30             18
2     0.7771         14                  90             21
3     0.7694         16                  90             12
4     0.7717         18                  150            9
5     0.7590         18                  30             15
6     0.7544         16                  110            9
7     0.7584         15                  30             9
8     0.7777         17                  150            15
9     0.7791         13                  150            15
10    0.7842         18                  70             24

CYBER ATTACK DETECTION OPTIMIZATION COMPLETE
Optimal Fitness Score: 0.7842
Optimized Hyperparameters: n_estimators=70, max_depth=24
Selected Cyber Features (18/30): ['duration', 'src_bytes', 'land', 'urgent', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_